# DeepONet vs SpectralSVR — antiderivative operator

Compares the DeepONet neural operator ([deepxde](https://deepxde.readthedocs.io), torch backend)
against **SpectralSVR** on the antiderivative operator: input = a function, output = its integral.

Both methods train on **identical data** — the same input functions sampled at the same sensor
locations — and are scored with the **same metric suite** (`utils.get_metrics`), so the comparison
is apples-to-apples. Every method is wrapped behind the shared `Operator` interface in
`SpectralSVR.operators`, so adding FNO / SNO / NSM later is just another adapter.

Requires the baselines extra: `uv sync --extra baselines`.


In [ ]:
import torch
import matplotlib.pyplot as plt

from SpectralSVR import FourierBasis, Antiderivative
from SpectralSVR.model import LSSVR
from SpectralSVR.operators import (
    OperatorDataset,
    SpectralSVROperator,
    DeepONetOperator,
    benchmark,
)

torch.set_default_dtype(torch.float64)


## Data

The antiderivative problem generates band-limited function pairs `(u, ut)` with `ut = du/dx`.
The operator we learn is the **integration** map `ut -> u`, so the input field is `ut` and the
output field is `u`. `OperatorDataset.from_fields` samples the input at a fixed sensor grid and
keeps the output as an evaluable spectral field.


In [ ]:
modes, n_train, n_test = 16, 400, 100
g = torch.Generator().manual_seed(0)
u, ut = Antiderivative().generate(FourierBasis, n_train + n_test, modes, generator=g, u0=0)

train = OperatorDataset.from_fields(ut[:n_train], u[:n_train], n_sensors=modes)
test = OperatorDataset.from_fields(ut[n_train:], u[n_train:], n_sensors=modes)
query = torch.linspace(0, 1, 100).unsqueeze(-1)  # (Q, 1) evaluation points
print('sensors', tuple(train.sensors.shape), 'train f', tuple(train.f.shape))


## Benchmark

`benchmark` fits each operator on the training split and scores it on the held-out test split at
the query points. DeepONet trains longer than the smoke test; tune `iterations` / layer widths.


In [ ]:
operators = [
    SpectralSVROperator(FourierBasis(), LSSVR(kernel='rbf', C=50.0)),
    DeepONetOperator(
        trunk=query,
        branch_layers=(64, 64),
        trunk_layers=(64, 64),
        iterations=20000,
        lr=1e-3,
        seed=0,
    ),
]
results = benchmark(operators, train, test, query)
results[['fit_seconds', 'rmse', 'mae', 'r2', 'rrse']].round(4)


## Predicted output functions

A few test functions: true antiderivative vs each method.


In [ ]:
target = test.targets(query).cpu()
preds = {op.name: op.predict(test.f, query).cpu() for op in operators}
xq = query.squeeze(-1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, i in zip(axes, range(3)):
    ax.plot(xq, target[i], 'k-', lw=2, label='true')
    for name, p in preds.items():
        ax.plot(xq, p[i], '--', label=name)
    ax.set_title(f'test function {i}')
    ax.legend()
fig.tight_layout()


## Extending

- **More test cases** (DeepONet paper, Lu et al. 2021): Burgers (the `Burgers` problem + exact
  solutions already exist), diffusion-reaction and pendulum/nonlinear-ODE (new `Problem` generators).
- **More methods**: add an adapter under `SpectralSVR/operators/` (FNO, SNO, NSM) implementing the
  `Operator` interface (`fit` / `predict`); it drops into `benchmark` unchanged.
